# JiuwenSwarm for Jupyter — Examples

This notebook walks through all examples from `docs/user/EXAMPLES.md`.
Each example is self-contained — run cells from top to bottom.

> **Note:** All examples work in any Jupyter environment with `jiuwenswarm-jupyter` installed.
> Example 3 additionally requires the JupyterLab TypeScript frontend to be built (see EXAMPLES.md).

## Prerequisites

```bash
pip install jiuwenswarm jiuwenswarm-jupyter
jiuwenswarm-init          # one-time setup — creates ~/.jiuwenswarm/
# then add your API key to ~/.jiuwenswarm/config/.env
```

No separate server to start. JiuwenSwarm runs entirely inside this kernel.

## Setup — synthetic dataset and extension load

All examples reference a customer churn dataset (`df`). The cell below creates
one synthetically so you can run this notebook without any external files.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 10_847

df = pd.DataFrame({
    "age": rng.integers(18, 75, n),
    "gender": rng.choice(["M", "F"], n),
    "tenure_months": rng.integers(1, 72, n),
    "monthly_charges": rng.uniform(20, 120, n).round(2),
    "total_charges": rng.uniform(0, 8000, n).round(2),
    "num_products": rng.integers(1, 6, n),
    "num_support_tickets": rng.integers(0, 15, n),
    "last_complaint_date": pd.date_range("2022-01-01", periods=n, freq="1h").strftime("%Y-%m-%d"),
    "churned": rng.choice([0, 1], n, p=[0.74, 0.26]),
})

# Introduce the two data quality issues mentioned in the examples
missing_idx = rng.choice(n, 11, replace=False)
df.loc[missing_idx, "total_charges"] = np.nan          # 11 missing values
# last_complaint_date is already a string (object dtype) — that is the issue

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing in total_charges: {df['total_charges'].isnull().sum()}")
print(f"last_complaint_date dtype: {df['last_complaint_date'].dtype}")
df.head(3)

Dataset shape: (10847, 9)
Columns: ['age', 'gender', 'tenure_months', 'monthly_charges', 'total_charges', 'num_products', 'num_support_tickets', 'last_complaint_date', 'churned']
Missing in total_charges: 11
last_complaint_date dtype: str


,age,gender,tenure_months,monthly_charges,total_charges,num_products,num_support_tickets,last_complaint_date,churned
0,23,M,28,119.25,5423.46,5,0,2022-01-01,0
1,62,F,4,27.59,5916.59,3,8,2022-01-01,0
2,55,F,55,85.80,2504.44,1,11,2022-01-01,0


In [2]:
%load_ext jiuwenswarm_jupyter

[JiuwenSwarm] Sidebar connected — JupyterLab comm active.
[jiuwenswarm] Restored session: jupyter_698d54c5ed02


---
## Example 1 — Understanding a dataset

**Who:** A data scientist who just received a CSV file from a client.

The agent reads `df` from the notebook namespace — you do not have to paste anything into the prompt.

**Expected output:**
> The dataset `df` has **10,847 rows and 21 columns**. ...
> **Data quality issues I can see:**
> 1. `total_charges` has 11 missing values
> 2. `last_complaint_date` is stored as a string
> 3. `monthly_charges` has a few extreme values

In [3]:
%%jiuwen
What does this dataset contain? How many rows and columns?
Are there any obvious data quality issues I should know about before I start?

C:\Workspace\openjiuwen\jiuwenswarm\.venv\Lib\site-packages\fastmcp\server\auth\providers\jwt.py:10: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken
C:\Workspace\openjiuwen\jiuwenswarm\.venv\Lib\site-packages\pydantic\main.py:263: DeprecationWarning: DefaultResponse is deprecated and will be removed in the future. Please use new config classes instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


2026-08-09 13:25:17 | sys_operation | default_trace_id | INFO | Deleted expired read-write lock database, lock_file=C:\Users\M00645~1\AppData\Local\Temp\openjiuwen-fs-rwlocks\532f1e05b657ceee1805a1e5a15291159fb2951c3f25d08f148dd16650f98425.db


---
## Example 2 — Writing code you can run

**Who:** Same data scientist, continuing from Example 1.

Use `--mode code` to tell the agent to focus on producing runnable code
rather than explanatory prose.

**Expected output:** A code block that fills missing `total_charges` values
and converts `last_complaint_date` to a proper datetime column.

In [4]:
%%jiuwen --mode code
Fix the two issues you mentioned:
1. Fill the missing total_charges values
2. Convert last_complaint_date to a proper datetime column
Write the code so I can run it right away.

In [5]:
# Apply the fix the agent suggested (paste the agent output here, or run this manually):
df["total_charges"] = df["total_charges"].fillna(0)
df["last_complaint_date"] = pd.to_datetime(df["last_complaint_date"], errors="coerce")

print("Missing values remaining:", df[["total_charges", "last_complaint_date"]].isnull().sum().to_dict())
print("last_complaint_date dtype:", df["last_complaint_date"].dtype)

Missing values remaining: {'total_charges': 0, 'last_complaint_date': 0}
last_complaint_date dtype: datetime64[us]


---
## Example 3 — Agent inserts the code cell directly (JupyterLab sidebar)

**Requires:** JupyterLab with the TypeScript frontend built and installed.

```bash
cd packages/frontend && npm install && npm run build
cd ../.. && pip install -e .
jupyter labextension develop --overwrite .
```

After restarting JupyterLab, click the JiuwenSwarm icon in the left sidebar
and type your request in the chat panel. The agent will insert code cells
directly into this notebook — no copy-pasting required.

> Example chat message: *"Fix the two data quality issues in df and put the code directly in my notebook"*

This uses the `insert_notebook_cell` tool called automatically by the agent.

---
## Example 4 — Multi-agent research (team mode)

**Who:** An ML engineer evaluating gradient boosting libraries.

`--mode team` spawns one sub-agent per research topic and runs them in parallel.

**Expected output:** A structured comparison of XGBoost, LightGBM, and CatBoost
covering speed, categorical handling, hyperparameter tuning, and community status.

In [ ]:
%%jiuwen --mode team
I need to pick between XGBoost, LightGBM, and CatBoost for a tabular classification task.
Please research each one in parallel and give me a comparison covering:
- Training speed on large datasets
- Handling of categorical features
- Ease of hyperparameter tuning
- Community support and maintenance status in 2025

---
## Example 5 — Named sessions: research in one thread, coding in another

**Who:** A researcher keeping literature review separate from implementation notes.

Each `--session` name maintains its own conversation history.
Switching names is like switching tabs — context never bleeds across sessions.

In [ ]:
%%jiuwen --session research
Find me 3 recent papers on contrastive learning for tabular data.
Summarise each in 2 sentences.

In [ ]:
%%jiuwen --session coding
Write a PyTorch Dataset class for my CSV file.
The file has columns: features (all float64) and target (0/1).

In [ ]:
%%jiuwen --session research
Based on the papers you found, which technique would be easiest to implement from scratch?

The last cell continues the **research** thread — the agent remembers the three papers
from the first research cell. The coding thread is unaffected.

---
## Example 6 — Working in PyCharm

All cell magics and notebook tools work normally in PyCharm Professional
(or any IDE with embedded Jupyter support). No extra setup required.

The only thing that does **not** work in PyCharm is the JupyterLab sidebar panel,
because that requires a browser-based JupyterLab frontend.

The cell below works identically in PyCharm, VS Code notebooks, and JupyterLab.

In [7]:
%%jiuwen --mode code
I have a list called `prices` that contains daily stock prices as floats.
Write a function that calculates the 20-day rolling average and returns a new list.

---
## Example 7 — Inspecting a trained model

**Who:** An ML engineer who just trained a model.

The agent reads `model`, `y_test`, and `y_pred` directly from the notebook namespace
using the `read_variable` tool. You do not paste any data into the prompt.

**Expected output:** Classification metrics + class imbalance analysis + suggested fix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Prepare features
X = df[["age", "tenure_months", "monthly_charges", "total_charges",
         "num_products", "num_support_tickets"]].values
y = df["churned"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
%%jiuwen
Look at the model variable and the y_test and y_pred arrays.
Calculate the main classification metrics and tell me if there is a class imbalance problem.

---
## Example 8 — Quick one-liner question

Use `%jiuwen` (line magic, single `%`) for a quick question without opening a full cell.
The answer appears immediately in the output area.

In [ ]:
%jiuwen What is the difference between fit() and fit_transform() in scikit-learn?

---
## Example 9 — Google Colab

All cell magics and notebook tools work in Google Colab without any extra setup.
The JupyterLab sidebar panel is not supported in Colab — use `%jiuwen_chat`
for an embedded chat UI in environments without the sidebar.

At the top of a Colab notebook:

```python
!pip install jiuwenswarm-jupyter jiuwenswarm -q
%load_ext jiuwenswarm_jupyter
```

Then use `%%jiuwen` normally:

```
%%jiuwen --mode code
I need to implement k-fold cross-validation from scratch without using sklearn.
The function should take X, y, a model, and k as input.
```

---
## Example 10 — Debugging with `%jiuwen_error`

When a cell raises an error, `%jiuwen_error` reads the full traceback and the
failing cell source automatically, then sends them to the agent for analysis.
You do not need to copy-paste the traceback.

Run the next two cells in order: first trigger the error, then call `%jiuwen_error`.

In [ ]:
# This cell intentionally raises a KeyError to demonstrate %jiuwen_error
label_map = {"product_type": {"A": 0, "B": 1}, "region": {"US": 0, "EU": 1}}

df["normalised"] = (df["monthly_charges"] - df["monthly_charges"].mean()) / df["monthly_charges"].std()
df["label_encoded"] = df["gender"].map(label_map["category"])   # KeyError: 'category'

In [ ]:
%jiuwen_error

You can also append a note on the same line:

```python
%jiuwen_error and also make sure NaN values in the column are handled
```

---
## Example 11 — Configuring defaults with `%jiuwen_config`

Set per-notebook defaults so you do not have to type `--mode` and `--timeout`
on every cell. Settings apply only to this notebook session.

In [ ]:
%jiuwen_config mode=team timeout=600

In [ ]:
# View current config at any time:
%jiuwen_config

In [ ]:
%%jiuwen --mode agent --timeout 30
Quick one-sentence answer: what is a p-value?

---
## Example 12 — Rewriting a cell with `replace_notebook_cell`

When the agent wants to improve existing code rather than add new code, it can call
`replace_notebook_cell`. In JupyterLab (with the sidebar connected) a diff dialog
appears; in other environments a coloured before/after diff is rendered in the output.

Ask the agent to rewrite a specific cell by referencing it directly in your prompt.

In [ ]:
# This cell has a bug: it normalises on the full column, leaking test statistics
def preprocess(df_in):
    df_out = df_in.copy()
    df_out["monthly_charges"] = (df_out["monthly_charges"] - df_out["monthly_charges"].mean()) / df_out["monthly_charges"].std()
    return df_out

# Call it:
df_processed = preprocess(df)
print(df_processed["monthly_charges"].describe())

In [ ]:
%%jiuwen --mode code
Look at the preprocess function I defined a few cells ago.
It normalises using the full column mean/std, which leaks test statistics.
Rewrite it so it accepts fit_mean and fit_std as optional parameters,
computing them from the data only if not provided.
Use replace_notebook_cell to update the cell directly.

---
## Bonus — ipywidgets control panel (`%jiuwen_panel`)

If `ipywidgets` is installed, `%jiuwen_panel` opens an interactive control
panel with mode/timeout sliders, a query textarea, and a Send button.
Useful in environments where typing magic syntax is inconvenient.

```bash
pip install ipywidgets
```

In [ ]:
%jiuwen_panel

---
## Bonus — Embedded full chat UI (`%jiuwen_chat`)

`%jiuwen_chat` embeds the same themed chat interface used by the JupyterLab
sidebar panel directly inside the cell output area.  No sidebar required.

**Most useful in:** Google Colab, Kaggle Notebooks, and classic Jupyter Notebook.
In JupyterLab the sidebar panel is the preferred interface, but `%jiuwen_chat`
works there too.

The iframe talks to the running kernel via the standard Jupyter comm channel —
same backend that powers the sidebar. All agent modes, session persistence, and
notebook context injection work normally.

In [6]:
# Embed the full chat UI in this cell's output area.
# Type a question in the text box and press Enter — responses stream in real time.
%jiuwen_chat

---
## Bonus — saving and restoring sessions (`%jiuwen_save`)

Save the current session ID to a JSON file so you can restore the conversation
after a kernel restart or in a different notebook.

In [ ]:
# Save the current session:
%jiuwen_save jiuwen_session.json

In [ ]:
# After a kernel restart — restore the session:
%jiuwen_save load jiuwen_session.json

---
## Summary — What works where

| Feature | JupyterLab | PyCharm | Google Colab | VS Code Notebooks |
|---|---|---|---|---|
| `%%jiuwen` / `%jiuwen` | Yes | Yes | Yes | Yes |
| `%jiuwen_error` | Yes | Yes | Yes | Yes |
| `%jiuwen_config` | Yes | Yes | Yes | Yes |
| `%jiuwen_export` | Yes | Yes | Yes | Yes |
| `%jiuwen_replay` | Yes | Yes | Yes | Yes |
| `%jiuwen_pin` / `%jiuwen_unpin` | Yes | Yes | Yes | Yes |
| `%jiuwen_panel` (ipywidgets) | Yes | Yes | Yes | Yes |
| `%jiuwen_chat` (embedded chat UI) | Yes | No | Yes | No |
| `read_variable()` | Yes | Yes | Yes | Yes |
| `read_notebook_cell()` | Yes | Yes | Yes | Yes |
| `insert_notebook_cell()` — display block | Yes | Yes | Yes | Yes |
| `replace_notebook_cell()` — diff in output | Yes | Yes | Yes | Yes |
| Sidebar panel + keyboard shortcuts | Yes | No | No | No |
| `insert_notebook_cell()` — in-notebook insert | Yes (sidebar) | No | No | No |
| `replace_notebook_cell()` — diff dialog | Yes (sidebar) | No | No | No |

See `docs/user/EXAMPLES.md` for the full narrative walkthrough of each example.